# Online Retail II Customer Sales Analysis

## DC-01 | Objectives

This notebook applies and validates data cleaning rules based on the findings from the Data Quality Assessment (DQA).

### Objectives

- Remove exact duplicate records.
- Evaluate and handle missing values.
- Validate data types and transaction quantities.
- Distinguish cancellation records from other adjustments.
- Validate zero and negative prices.
- Prepare and export a reproducible cleaned dataset for downstream analysis.

### Cleaning Principles

1. Preserve the original raw dataset.
2. Apply cleaning rules in a documented sequence.
3. Track the number of records removed at each step.
4. Preserve relevant business transactions where appropriate.
5. Apply analysis-specific filters separately from general data cleaning.

## DC-02 | Environment & Data Loading

This section imports the required libraries, loads the original Online Retail II dataset, and verifies its structure before cleaning.

The raw dataset is preserved, and all cleaning operations are performed on a separate working DataFrame.

In [1]:
# DC-02-01: Import Libraries

import pandas as pd
import numpy as np

In [2]:
# DC-02-02: Load Raw Dataset

file_path = "../data/raw/online_retail_II.csv"

df_raw = pd.read_csv(file_path)

# Create a working copy for cleaning
df = df_raw.copy()

print("Raw dataset loaded successfully.")

Raw dataset loaded successfully.


In [3]:
# DC-02-03: Initial Dataset Verification

print(f"Total Rows: {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]}")

print("\nFirst 5 Rows:")
display(df.head())

print("\nDataset Information:")
df.info()

Total Rows: 1,067,371
Total Columns: 8

First 5 Rows:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom



Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


## DC-03 | Duplicate Removal

This section removes exact duplicate records identified during the Data Quality Assessment.

Duplicate detection is performed across all columns to maintain consistency with the DQA methodology.

In [4]:
# DC-03-01: Identify and Remove Exact Duplicates

rows_before = len(df)

duplicate_count = df.duplicated().sum()

df = df.drop_duplicates().copy()

rows_after = len(df)
removed_count = rows_before - rows_after

print(f"Rows Before: {rows_before:,}")
print(f"Duplicate Records: {duplicate_count:,}")
print(f"Rows After: {rows_after:,}")
print(f"Removed Records: {removed_count:,}")

assert duplicate_count == removed_count
assert df.duplicated().sum() == 0

Rows Before: 1,067,371
Duplicate Records: 34,335
Rows After: 1,033,036
Removed Records: 34,335


### Interpretation

- A total of 34,335 exact duplicate records (3.22%) were identified in the raw dataset.
- After duplicate removal, 1,033,036 records remain.
- Duplicate removal was performed across all columns.
- The cleaned working DataFrame is used for subsequent cleaning steps.

**Cleaning Implication:** Removing exact duplicates helps prevent potential overcounting in downstream transaction and revenue analysis.

## DC-04 | Missing Value Handling

This section evaluates missing values after duplicate removal and applies cleaning rules according to the analytical purpose.

In [5]:
# DC-04-01: Missing Value Summary After Deduplication

missing_after_dedup = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage (%)": (
        df.isna().mean() * 100
    ).round(2)
})

display(
    missing_after_dedup.sort_values(
        "Missing Count",
        ascending=False
    )
)

print(f"Current Rows: {len(df):,}")

,Missing Count,Missing Percentage (%)
Customer ID,235151,22.76
Description,4275,0.41
StockCode,0,0.00
Invoice,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Country,0,0.00


Current Rows: 1,033,036


In [6]:
# DC-04-02: Description Missing Value Investigation

missing_desc = df[df["Description"].isna()].copy()

desc_summary = pd.Series({
    "Missing Description Records": len(missing_desc),
    "Price = 0": (missing_desc["Price"] == 0).sum(),
    "Price > 0": (missing_desc["Price"] > 0).sum(),
    "Price < 0": (missing_desc["Price"] < 0).sum(),
    "Quantity < 0": (missing_desc["Quantity"] < 0).sum(),
    "Customer ID Missing": missing_desc["Customer ID"].isna().sum()
})

display(desc_summary.to_frame("Record Count"))

print("\nSample Records:")
display(missing_desc.head(10))

,Record Count
Missing Description Records,4275
Price = 0,4275
Price > 0,0
Price < 0,0
Quantity < 0,2633
Customer ID Missing,4275



Sample Records:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom
6378,489882,35751C,NaN,12,2009-12-02 16:22:00,0.0,NaN,United Kingdom
6555,489898,79323G,NaN,954,2009-12-03 09:40:00,0.0,NaN,United Kingdom
6576,489901,21098,NaN,-200,2009-12-03 09:47:00,0.0,NaN,United Kingdom
6581,489903,21166,NaN,48,2009-12-03 09:57:00,0.0,NaN,United Kingdom


### DC-04-03 | Missing Value Treatment

Records with missing product descriptions are excluded from the working dataset because all such records have zero prices and missing customer identifiers.

Records with missing Customer ID but valid product descriptions are retained at this stage for potential aggregate sales analysis.


In [7]:
# DC-04-03: Remove Missing Description Records

rows_before = len(df)

df = df.dropna(subset=["Description"]).copy()

rows_after = len(df)
removed_description = rows_before - rows_after

print(f"Rows Before: {rows_before:,}")
print(f"Description Missing Removed: {removed_description:,}")
print(f"Rows After: {rows_after:,}")

print("\nRemaining Missing Values:")
display(df.isna().sum())

assert df["Description"].isna().sum() == 0

Rows Before: 1,033,036
Description Missing Removed: 4,275
Rows After: 1,028,761

Remaining Missing Values:


Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer ID    230876
Country             0
dtype: int64

## DC-05 | Data Type Conversion

This section converts transaction dates and customer identifiers into appropriate data types for downstream analysis.

- **InvoiceDate:** Convert from string to datetime for time-series analysis and RFM Recency calculation.
- **Customer ID:** Convert from float64 to nullable string to preserve missing values and prevent customer identifiers from being treated as continuous numerical variables.

In [8]:
# DC-05-01: Convert Data Types

# Convert invoice dates
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce"
)

# Convert customer identifiers to nullable string
df["Customer ID"] = (
    df["Customer ID"]
    .astype("Int64")
    .astype("string")
)

# Validate conversions
print("Updated Data Types:")
display(df.dtypes)

print("\nInvalid InvoiceDate Records:",
      df["InvoiceDate"].isna().sum())

print("Missing Customer ID Records:",
      df["Customer ID"].isna().sum())

print("\nSample Converted Data:")
display(df[["InvoiceDate", "Customer ID"]].head())

Updated Data Types:


Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID            string
Country                   str
dtype: object


Invalid InvoiceDate Records: 0
Missing Customer ID Records: 230876

Sample Converted Data:


,InvoiceDate,Customer ID
0,2009-12-01 07:45:00,13085
1,2009-12-01 07:45:00,13085
2,2009-12-01 07:45:00,13085
3,2009-12-01 07:45:00,13085
4,2009-12-01 07:45:00,13085


### Interpretation

- **InvoiceDate:** Successfully converted from string to datetime64[us], enabling time-series and recency calculations.
- **Customer ID:** Converted from float64 to nullable string, preserving customer identifiers without decimal formatting.
- **Date Validation:** No invalid dates were introduced during conversion.
- **Missing Values:** All 230,876 missing Customer ID records were preserved.

**Cleaning Implication:** Data types are now standardized for downstream analysis while retaining records that may be useful for aggregate sales analysis.

## DC-06 | Quantity & Adjustment Handling

This section examines negative quantities, cancellation indicators, and potential adjustment records after duplicate and missing-value cleaning.

The objective is to distinguish transaction types before defining the final dataset for downstream sales analysis.

In [9]:
# DC-06-01: Quantity and Cancellation Assessment

negative_mask = df["Quantity"] < 0

cancel_mask = (
    df["Invoice"]
    .astype("string")
    .str.upper()
    .str.startswith("C")
    .fillna(False)
)

quantity_summary = pd.Series({
    "Total Records": len(df),
    "Negative Quantity": negative_mask.sum(),
    "Zero Quantity": (df["Quantity"] == 0).sum(),
    "Cancellation Flag": cancel_mask.sum(),
    "Negative + Cancellation": (
        negative_mask & cancel_mask
    ).sum(),
    "Negative without Cancellation": (
        negative_mask & ~cancel_mask
    ).sum(),
    "Cancellation without Negative": (
        ~negative_mask & cancel_mask
    ).sum()
})

display(quantity_summary.to_frame("Record Count"))

,Record Count
Total Records,1028761
Negative Quantity,19863
Zero Quantity,0
Cancellation Flag,19104
Negative + Cancellation,19103
Negative without Cancellation,760
Cancellation without Negative,1


In [10]:
# DC-06-02: Investigate Negative Quantities Without Cancellation Flag

negative_without_c = df[
    (df["Quantity"] < 0) &
    (~cancel_mask)
].copy()

print(f"Records to Investigate: {len(negative_without_c):,}")

# Summary of transaction characteristics
investigation_summary = pd.Series({
    "Total Records": len(negative_without_c),
    "Price = 0": (negative_without_c["Price"] == 0).sum(),
    "Price > 0": (negative_without_c["Price"] > 0).sum(),
    "Price < 0": (negative_without_c["Price"] < 0).sum(),
    "Customer ID Missing": negative_without_c["Customer ID"].isna().sum(),
    "Unique StockCodes": negative_without_c["StockCode"].nunique(),
    "Unique Descriptions": negative_without_c["Description"].nunique()
})

display(investigation_summary.to_frame("Record Count"))

print("\nTop 20 Descriptions:")
display(
    negative_without_c["Description"]
    .value_counts()
    .head(20)
)

print("\nTop 20 StockCodes:")
display(
    negative_without_c["StockCode"]
    .value_counts()
    .head(20)
)

print("\nSample Records:")
display(
    negative_without_c[
        ["Invoice", "StockCode", "Description",
         "Quantity", "Price", "Customer ID"]
    ].head(20)
)

Records to Investigate: 760


,Record Count
Total Records,760
Price = 0,760
Price > 0,0
Price < 0,0
Customer ID Missing,760
Unique StockCodes,626
Unique Descriptions,222



Top 20 Descriptions:


Description
check                     121
damages                    83
?                          81
damaged                    78
missing                    27
sold as set on dotcom      20
Damaged                    17
smashed                     9
thrown away                 9
Unsaleable, destroyed.      9
dotcom                      8
damages?                    7
??                          7
crushed                     6
given away                  6
MIA                         5
Damages                     5
counted                     5
checked                     5
wet damaged                 5
Name: count, dtype: int64


Top 20 StockCodes:


StockCode
22423     10
84016      5
20852      5
85175      5
47566B     4
22719      4
21843      4
85172      4
21830      4
72802C     4
21100      3
82494L     3
21463      3
79000      3
21735      3
20713      3
37479P     3
82582      3
84406B     3
46000S     3
Name: count, dtype: int64


Sample Records:


,Invoice,StockCode,Description,Quantity,Price,Customer ID
263,489464,21733,85123a mixed,-96,0.0,<NA>
283,489463,71477,short,-240,0.0,<NA>
284,489467,85123A,21733 mixed,-192,0.0,<NA>
3162,489660,35956,lost,-1043,0.0,<NA>
3168,489663,35605A,damages,-117,0.0,<NA>
4538,489820,21133,invcd as 84879?,-720,0.0,<NA>
6556,489899,79323GR,sold as gold,-954,0.0,<NA>
6911,490007,84347,21494,-720,0.0,<NA>
9308,490130,21493,lost?,-600,0.0,<NA>
17427,490765,21450,damaged,-31,0.0,<NA>


### DC-06-03 | Adjustment Classification & Treatment

The 760 negative-quantity records without cancellation flags all have zero prices and missing customer identifiers.

Their descriptions suggest inventory adjustments, damaged goods, or other non-sales activities, although the exact business reason cannot be confirmed for every record.

These records are classified as non-sales candidates and excluded from the sales-oriented working dataset while being preserved separately for auditability.

In [11]:
# DC-06-03: Classify and Separate Non-Sales Candidates

# Preserve the 760 records separately
non_sales_candidates = negative_without_c.copy()

# Create a sales-oriented working dataset
df_sales_base = df.drop(
    index=non_sales_candidates.index
).copy()

print(f"Records Before: {len(df):,}")
print(f"Non-Sales Candidates: {len(non_sales_candidates):,}")
print(f"Sales Base Records: {len(df_sales_base):,}")

assert len(non_sales_candidates) == 760
assert len(df_sales_base) == len(df) - 760

Records Before: 1,028,761
Non-Sales Candidates: 760
Sales Base Records: 1,028,001


### DC-06-04 | Post-Separation Validation

This section verifies that non-sales candidates were separated correctly and examines the remaining negative quantities and cancellation indicators in the sales-oriented dataset.

In [12]:
# DC-06-04: Post-Separation Validation

remaining_negative = df_sales_base["Quantity"] < 0

remaining_cancel = (
    df_sales_base["Invoice"]
    .astype("string")
    .str.upper()
    .str.startswith("C")
    .fillna(False)
)

validation_summary = pd.Series({
    "Sales Base Records": len(df_sales_base),
    "Remaining Negative Quantity": remaining_negative.sum(),
    "Remaining Zero Quantity": (df_sales_base["Quantity"] == 0).sum(),
    "Remaining Cancellation Flags": remaining_cancel.sum(),
    "Negative without Cancellation": (
        remaining_negative & ~remaining_cancel
    ).sum(),
    "Remaining Non-Positive Prices": (
        df_sales_base["Price"] <= 0
    ).sum()
})

display(validation_summary.to_frame("Record Count"))

# Verify separation
assert len(df_sales_base) + len(non_sales_candidates) == len(df)
assert (remaining_negative & ~remaining_cancel).sum() == 0

,Record Count
Sales Base Records,1028001
Remaining Negative Quantity,19103
Remaining Zero Quantity,0
Remaining Cancellation Flags,19104
Negative without Cancellation,0
Remaining Non-Positive Prices,984


## DC-07 | Price & Invoice Validation

This section examines non-positive prices and cancellation indicators in the sales-oriented working dataset.

The objective is to establish explicit transaction filters for positive-sales analysis while preserving the broader cleaned dataset.

In [13]:
# DC-07-01: Price and Invoice Validation

negative_price_mask = df_sales_base["Price"] < 0
zero_price_mask = df_sales_base["Price"] == 0
positive_price_mask = df_sales_base["Price"] > 0

cancel_mask_sales = (
    df_sales_base["Invoice"]
    .astype("string")
    .str.upper()
    .str.startswith("C")
    .fillna(False)
)

price_invoice_summary = pd.Series({
    "Total Sales Base Records": len(df_sales_base),
    "Negative Price": negative_price_mask.sum(),
    "Zero Price": zero_price_mask.sum(),
    "Positive Price": positive_price_mask.sum(),
    "Cancellation Flag": cancel_mask_sales.sum(),
    "Non-Positive Price + Cancellation": (
        (negative_price_mask | zero_price_mask) & cancel_mask_sales
    ).sum()
})

display(price_invoice_summary.to_frame("Record Count"))

print("\nSample Non-Positive Price Records:")
display(
    df_sales_base[
        negative_price_mask | zero_price_mask
    ].head(10)
)

print("\nCancellation Flag Without Negative Quantity:")
display(
    df_sales_base[
        cancel_mask_sales & (df_sales_base["Quantity"] >= 0)
    ]
)

,Record Count
Total Sales Base Records,1028001
Negative Price,5
Zero Price,979
Positive Price,1027017
Cancellation Flag,19104
Non-Positive Price + Cancellation,0



Sample Non-Positive Price Records:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126,United Kingdom
5904,489861,DOT,DOTCOM POSTAGE,1,2009-12-02 14:50:00,0.0,<NA>,United Kingdom
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658,United Kingdom
16107,490727,M,Manual,1,2009-12-07 16:38:00,0.0,17231,United Kingdom
18738,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,2009-12-08 15:25:00,0.0,14108,United Kingdom
18739,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-08 15:25:00,0.0,14108,United Kingdom
31993,491971,85042,ANTIQUE LILY FAIRY LIGHTS,1,2009-12-14 18:37:00,0.0,<NA>,United Kingdom
32916,492079,85042,ANTIQUE LILY FAIRY LIGHTS,8,2009-12-15 13:49:00,0.0,15070,United Kingdom
40101,492760,21143,ANTIQUE GLASS HEART DECORATION,12,2009-12-18 14:22:00,0.0,18071,United Kingdom
47126,493761,79320,FLAMINGO LIGHTS,24,2010-01-06 14:54:00,0.0,14258,United Kingdom



Cancellation Flag Without Negative Quantity:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
76799,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,<NA>,United Kingdom


### DC-07-02 | Positive Sales Dataset Preparation

This section creates an analysis-specific dataset containing positive-quantity, positive-price transactions without cancellation indicators.

The broader cleaned dataset is preserved to support auditability and alternative analytical objectives.

In [14]:
# DC-07-02: Create Positive Sales Dataset

positive_sales_mask = (
    (df_sales_base["Quantity"] > 0) &
    (df_sales_base["Price"] > 0) &
    (~cancel_mask_sales)
)

df_positive_sales = df_sales_base.loc[
    positive_sales_mask
].copy()

# Calculate sales revenue
df_positive_sales["Sales"] = (
    df_positive_sales["Quantity"] *
    df_positive_sales["Price"]
)

print(f"Sales Base Records: {len(df_sales_base):,}")
print(f"Positive Sales Records: {len(df_positive_sales):,}")
print(f"Excluded Records: {len(df_sales_base) - len(df_positive_sales):,}")
print(f"Total Positive Sales Revenue: {df_positive_sales['Sales'].sum():,.2f}")

# Validate dataset
assert (df_positive_sales["Quantity"] > 0).all()
assert (df_positive_sales["Price"] > 0).all()
assert not df_positive_sales["Invoice"].astype("string").str.upper().str.startswith("C").any()

Sales Base Records: 1,028,001
Positive Sales Records: 1,007,913
Excluded Records: 20,088
Total Positive Sales Revenue: 20,476,260.45


### Interpretation

- **Positive Sales Records:** 1,007,913 transaction lines satisfy the positive quantity, positive price, and non-cancellation criteria.
- **Excluded Records:** 20,088 transaction lines were excluded from the sales-oriented dataset due to cancellation indicators or non-positive prices.
- **Total Positive Sales Revenue:** £20,476,260.45.
- **Data Preservation:** The broader cleaned dataset and non-sales candidates remain available separately.

**Cleaning Implication:** The positive-sales dataset provides a consistent basis for downstream sales performance analysis. Customer-level RFM analysis will additionally require valid Customer IDs.

### Business Interpretation

Negative quantity records were devided into two categories:

- Customer return or calcellation transactions (Invoice begins with **C**)
- Interanl inventory adjustment records (e.g., *damaged*, *lost*, *short*, *bad quality*)

Return transactions were retained becaue they represent valid business events.

Inventory adjustment records were removed because they do not represent customer purchases and are outside the scope of this sales analysis project.

## DC-08 | Clean Dataset Export

This section prepares and exports cleaned datasets for downstream analysis.

### Export Strategy

- **General Cleaned Dataset:** Preserves the broader transaction data after duplicate removal, missing Description handling, and data type conversion.
- **Positive Sales Dataset:** Contains transactions with positive quantities, positive prices, and no cancellation flags.
- **RFM Dataset:** Contains positive sales transactions with valid Customer IDs.
- **Non-Sales Candidates:** Preserves negative-quantity, zero-price records without cancellation flags for auditability.

All datasets will be saved in the `data/processed/` directory.

In [15]:
# DC-08-01: Prepare RFM Dataset

df_rfm = df_positive_sales.dropna(
    subset=["Customer ID"]
).copy()

print(f"Positive Sales Records: {len(df_positive_sales):,}")
print(f"RFM Records: {len(df_rfm):,}")
print(
    f"Excluded Missing Customer IDs: "
    f"{len(df_positive_sales) - len(df_rfm):,}"
)

print(f"Unique Customers: {df_rfm['Customer ID'].nunique():,}")

# Validation
assert df_rfm["Customer ID"].isna().sum() == 0
assert (df_rfm["Quantity"] > 0).all()
assert (df_rfm["Price"] > 0).all()

Positive Sales Records: 1,007,913
RFM Records: 779,425
Excluded Missing Customer IDs: 228,488
Unique Customers: 5,878


### DC-08-01 | Interpretation

- **Positive Sales Records:** 1,007,913 transaction lines.
- **RFM Records:** 779,425 transaction lines (77.33%).
- **Missing Customer ID Exclusions:** 228,488 transaction lines (22.67%).
- **Unique Customers:** 5,878.

The RFM dataset retains only positive sales transactions with valid customer identifiers.

**Analytical Implication:** EDA uses the broader positive-sales dataset, while RFM analysis uses the customer-identifiable subset. Therefore, revenue totals and transaction counts may differ between the two analyses.

In [16]:
# DC-08-02: Inspect Existing Export Files

from pathlib import Path

output_dir = Path("../data/processed")

print("Current Working Directory:")
print(Path.cwd().resolve())

print("\nTarget Output Directory:")
print(output_dir.resolve())

print("\nExisting CSV Files:")

if output_dir.exists():
    csv_files = sorted(output_dir.glob("*.csv"))

    if csv_files:
        for file in csv_files:
            print(
                f"{file.name} | "
                f"{file.stat().st_size / (1024**2):.2f} MB"
            )
    else:
        print("No CSV files found.")
else:
    print("Output directory does not exist.")

Current Working Directory:
C:\Users\user\OneDrive\Documents\ecommerce-customer_sales-analysis-updated\ecommerce--customer-sales-analysis\notebooks

Target Output Directory:
C:\Users\user\OneDrive\Documents\ecommerce-customer_sales-analysis-updated\ecommerce--customer-sales-analysis\data\processed

Existing CSV Files:
inventory_adjustments.csv | 0.05 MB
non_sales_candidates_v2.csv | 0.05 MB
online_retail_clean.csv | 88.23 MB
online_retail_cleaned_v2.csv | 86.76 MB
online_retail_positive_sales_v2.csv | 91.97 MB
online_retail_rfm_v2.csv | 72.15 MB


### DC-08-03 | Export Cleaned Datasets

The validated datasets are exported as versioned CSV files.

Existing processed files are preserved to prevent unintended changes to downstream EDA and RFM notebooks.

In [17]:
# DC-08-03: Export Versioned Datasets

from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

datasets = {
    "online_retail_cleaned_v2.csv": df,
    "online_retail_positive_sales_v2.csv": df_positive_sales,
    "online_retail_rfm_v2.csv": df_rfm,
    "non_sales_candidates_v2.csv": non_sales_candidates
}

for filename, dataset in datasets.items():
    output_path = output_dir / filename

    if output_path.exists():
        print(f"SKIPPED (already exists): {filename}")
        continue

    dataset.to_csv(
        output_path,
        index=False
    )

    print(
        f"SAVED: {filename} | "
        f"Rows: {len(dataset):,} | "
        f"Columns: {len(dataset.columns)}"
    )

SKIPPED (already exists): online_retail_cleaned_v2.csv
SKIPPED (already exists): online_retail_positive_sales_v2.csv
SKIPPED (already exists): online_retail_rfm_v2.csv
SKIPPED (already exists): non_sales_candidates_v2.csv


In [18]:
# DC-08-04: Verify Exported Datasets

for filename, dataset in datasets.items():
    file_path = output_dir / filename

    assert file_path.exists(), f"Missing file: {filename}"

    saved_df = pd.read_csv(file_path)

    assert len(saved_df) == len(dataset), (
        f"Row count mismatch: {filename}"
    )

    assert len(saved_df.columns) == len(dataset.columns), (
        f"Column count mismatch: {filename}"
    )

    print(
        f"PASS | {filename} | "
        f"Rows: {len(saved_df):,} | "
        f"Columns: {len(saved_df.columns)} | "
        f"Size: {file_path.stat().st_size / (1024**2):.2f} MB"
    )

PASS | online_retail_cleaned_v2.csv | Rows: 1,028,761 | Columns: 8 | Size: 86.76 MB


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_24504\3951683454.py:8: DtypeWarning: Columns (0: Invoice) have mixed types. Specify dtype option on import or set low_memory=False.
  saved_df = pd.read_csv(file_path)


PASS | online_retail_positive_sales_v2.csv | Rows: 1,007,913 | Columns: 9 | Size: 91.97 MB
PASS | online_retail_rfm_v2.csv | Rows: 779,425 | Columns: 9 | Size: 72.15 MB
PASS | non_sales_candidates_v2.csv | Rows: 760 | Columns: 8 | Size: 0.05 MB


## DC-09 | Final QA & Cleaning Summary

This section validates the final cleaned datasets and documents the data cleaning results.

### Objectives

- Verify the cleaning process from raw data to final datasets.
- Compare legacy and versioned cleaned datasets.
- Validate transaction counts and sales revenue.
- Confirm dataset readiness for EDA and RFM analysis.

In [19]:
# DC-09-01: Cleaning Pipeline Summary

cleaning_summary = pd.DataFrame({
    "Stage": [
        "Raw Dataset",
        "After Duplicate Removal",
        "After Missing Description Removal",
        "Sales Base (Non-Sales Candidates Excluded)",
        "Positive Sales Dataset",
        "RFM Dataset"
    ],
    "Records": [
        len(df_raw),
        1_033_036,
        len(df),
        len(df_sales_base),
        len(df_positive_sales),
        len(df_rfm)
    ]
})

cleaning_summary["Change from Previous Stage"] = (
    cleaning_summary["Records"].diff().fillna(0).astype(int)
)

display(cleaning_summary)

print(
    f"\nTotal Positive Sales Revenue: "
    f"{df_positive_sales['Sales'].sum():,.2f}"
)

print(
    f"Unique RFM Customers: "
    f"{df_rfm['Customer ID'].nunique():,}"
)

assert len(df_raw) == 1_067_371
assert len(df_positive_sales) == 1_007_913
assert len(df_rfm) == 779_425

,Stage,Records,Change from Previous Stage
0,Raw Dataset,1067371,0
1,After Duplicate Removal,1033036,-34335
2,After Missing Description Removal,1028761,-4275
3,Sales Base (Non-Sales Candidates Excluded),1028001,-760
4,Positive Sales Dataset,1007913,-20088
5,RFM Dataset,779425,-228488



Total Positive Sales Revenue: 20,476,260.45
Unique RFM Customers: 5,878


### DC-09-02 | Legacy vs. V2 Dataset Comparison

This section compares the legacy cleaned dataset with the newly validated V2 dataset.

The objective is to identify differences in data structure, record counts, missing values, and transaction characteristics before updating downstream notebooks.

No existing files will be modified during this comparison.

In [20]:
# DC-09-02: Compare Legacy and V2 Cleaned Datasets

legacy_path = output_dir / "online_retail_clean.csv"
v2_path = output_dir / "online_retail_cleaned_v2.csv"

legacy_df = pd.read_csv(legacy_path, low_memory=False)
v2_df = pd.read_csv(v2_path, low_memory=False)

comparison = pd.DataFrame({
    "Metric": [
        "Total Records",
        "Total Columns",
        "Missing Customer IDs",
        "Missing Descriptions",
        "Negative Quantities",
        "Zero Prices",
        "Negative Prices",
        "Unique Invoices"
    ],
    "Legacy": [
        len(legacy_df),
        len(legacy_df.columns),
        legacy_df["Customer ID"].isna().sum(),
        legacy_df["Description"].isna().sum(),
        (legacy_df["Quantity"] < 0).sum(),
        (legacy_df["Price"] == 0).sum(),
        (legacy_df["Price"] < 0).sum(),
        legacy_df["Invoice"].nunique()
    ],
    "V2": [
        len(v2_df),
        len(v2_df.columns),
        v2_df["Customer ID"].isna().sum(),
        v2_df["Description"].isna().sum(),
        (v2_df["Quantity"] < 0).sum(),
        (v2_df["Price"] == 0).sum(),
        (v2_df["Price"] < 0).sum(),
        v2_df["Invoice"].nunique()
    ]
})

comparison["Difference (V2 - Legacy)"] = (
    comparison["V2"] - comparison["Legacy"]
)

display(comparison)

print("\nLegacy Columns:", legacy_df.columns.tolist())
print("V2 Columns:", v2_df.columns.tolist())

print("\nLegacy Data Types:")
print(legacy_df.dtypes)

print("\nV2 Data Types:")
print(v2_df.dtypes)

,Metric,Legacy,V2,Difference (V2 - Legacy)
0,Total Records,1028001,1028761,760
1,Total Columns,8,8,0
2,Missing Customer IDs,230116,230876,760
3,Missing Descriptions,0,0,0
4,Negative Quantities,19103,19863,760
5,Zero Prices,979,1739,760
6,Negative Prices,5,5,0
7,Unique Invoices,48593,49353,760



Legacy Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
V2 Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Legacy Data Types:
Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object

V2 Data Types:
Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object


### DC-09-03 | Row-Level Consistency Validation

This section verifies whether the legacy cleaned dataset
contains the same transaction records as the newly created
sales-base dataset.

The comparison uses normalized data types and row-level
matching rather than aggregate counts alone.

In [21]:
# DC-09-03: Row-Level Consistency Validation

from pandas.testing import assert_frame_equal

# Normalize data types for comparison
def normalize_dataset(data):
    result = data.copy()

    result["InvoiceDate"] = pd.to_datetime(
        result["InvoiceDate"], errors="raise"
    )

    result["Customer ID"] = (
        pd.to_numeric(
            result["Customer ID"], errors="coerce"
        )
        .astype("Int64")
        .astype("string")
    )

    return result

legacy_normalized = normalize_dataset(legacy_df)
sales_base_normalized = normalize_dataset(df_sales_base)

# Compare row contents regardless of row order
columns = sorted(legacy_normalized.columns.tolist())

legacy_sorted = (
    legacy_normalized[columns]
    .sort_values(columns, na_position="last")
    .reset_index(drop=True)
)

sales_base_sorted = (
    sales_base_normalized[columns]
    .sort_values(columns, na_position="last")
    .reset_index(drop=True)
)

try:
    assert_frame_equal(
        legacy_sorted,
        sales_base_sorted,
        check_dtype=False
    )

    print("PASS: Legacy and Sales Base datasets match.")

except AssertionError as e:
    print("FAIL: Dataset differences detected.")
    print(str(e)[:1000])

PASS: Legacy and Sales Base datasets match.


### DC-09-03 | Interpretation

- **Legacy Dataset:** 1,028,001 transaction records.
- **V2 Sales Base:** 1,028,001 transaction records.
- **Row-Level Validation:** PASS.
- **Dataset Consistency:** Both datasets contain identical transaction records after data type normalization and row sorting.

**Key Finding:** The legacy cleaned dataset is consistent with the newly reconstructed sales-base dataset. The 760-record difference between the legacy dataset and the broader V2 cleaned dataset is attributable to the separately preserved non-sales candidates.

**Implication:** The existing cleaned dataset can be retained for backward compatibility. Downstream EDA and RFM notebooks should be reviewed to confirm their transaction filters and analytical results.

### DC-09-04 | Final Cleaning Summary

#### Data Cleaning Results

| Stage | Records |
|---|---:|
| Raw Dataset | 1,067,371 |
| After Duplicate Removal | 1,033,036 |
| After Missing Description Removal | 1,028,761 |
| Non-Sales Candidates | 760 |
| Sales Base Dataset | 1,028,001 |
| Positive Sales Dataset | 1,007,913 |
| RFM Dataset | 779,425 |

#### Key Findings

1. Removed 34,335 exact duplicate records.
2. Removed 4,275 records with missing product descriptions.
3. Identified and separately preserved 760 non-sales candidates.
4. Excluded cancellation transactions and non-positive prices from positive-sales analysis.
5. Prepared a customer-identifiable dataset for RFM analysis.
6. Confirmed row-level consistency between the legacy cleaned dataset and the reconstructed sales-base dataset.

#### Final Analytical Outputs

- **Positive Sales Revenue:** £20,476,260.45
- **Positive Sales Records:** 1,007,913
- **RFM Transaction Records:** 779,425
- **Unique RFM Customers:** 5,878

#### Limitations

- Non-sales candidates were classified using transaction characteristics and descriptions rather than confirmed business records.
- Missing Customer IDs limit customer-level analysis.
- Positive-sales revenue excludes cancellations and returns and should not be interpreted as net revenue.
- CSV exports do not preserve pandas data types automatically.

#### Next Steps

- Review Notebook 03: Exploratory Data Analysis.
- Review Notebook 04: RFM Analysis.
- Validate consistency between analytical results and the final presentation.
- Update GitHub documentation after notebook QA.